Classification - 다중분류 -> categorical_crossentropy / 원핫인코딩 사용

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abalone.csv")

if "id" in df.columns:
    df = df.drop("id", axis=1)

df = pd.get_dummies(df, columns=["Sex"], dtype=int)

X = df.drop("Rings", axis=1).values
y_raw = df["Rings"].values

y_zero_based = y_raw - 1
num_classes = 29

y = tf.keras.utils.to_categorical(y_zero_based, num_classes=num_classes)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=None
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print("\n--- Model Summary ---")
model.summary()

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

accuracy = accuracy_score(y_true, y_pred)
print("\nAccuracy =", accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))


--- Model Summary ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 29)             │           957 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,741 (14.61 KB)

 Trainable params: 3,741 (14.61 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.1748 - loss: 2.8607 - val_accuracy: 0.2197 - val_loss: 2.3662
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2324 - loss: 2.2911 - val_accuracy: 0.2422 - val_loss: 2.1156
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2504 - loss: 2.1441 - val_accuracy: 0.2481 - val_loss: 2.0514
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2586 - loss: 2.0902 - val_accuracy: 0.2556 - val_loss: 2.0201
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2650 - loss: 2.0586 - val_accuracy: 0.2601 - val_loss: 1.9978
Epoch 6/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2695 - loss: 2.0348 - val_accuracy: 0.2601 - val_loss: 1.9794
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2691 - loss: 2.0148 - val_accuracy: 0.2661 - val_loss: 1.9647
Epoch 8/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2743 - loss: 1.9977 - val_accuracy: 0.2676 - v

타겟인 Rings는 범주이기 때문에 Regression이 더 적합한 것 같음.

Regression -> mse / None

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abalone.csv")

print("--- 데이터 확인 ---")
print(df.head())
print(f"\nData Shape: {df.shape}")

if "id" in df.columns:
    df = df.drop("id", axis=1)

df = pd.get_dummies(df, columns=["Sex"], dtype=int)

X = df.drop("Rings", axis=1).values
y = df["Rings"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation=None)
])

model.compile(
    optimizer="adam",
    loss="mse"
)

print("\n--- Model Summary ---")
model.summary()

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

y_pred = model.predict(X_test).flatten()

mse = mean_squared_error(y_test, y_pred)
print("\n--- Model Evaluation ---")
print(f"Test MSE: {mse:.4f}")

--- 데이터 확인 ---
   id Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  \
0   0   M   0.455     0.365   0.095        0.5140          0.2245   
1   1   M   0.350     0.265   0.090        0.2255          0.0995   
2   2   F   0.530     0.420   0.135        0.6770          0.2565   
3   3   M   0.440     0.365   0.125        0.5160          0.2155   
4   4   I   0.330     0.255   0.080        0.2050          0.0895   

   Viscera_weight  Shell_weight  Rings  
0          0.1010         0.150     15  
1          0.0485         0.070      7  
2          0.1415         0.210      9  
3          0.1140         0.155     10  
4          0.0395         0.055      7  

Data Shape: (4177, 10)

--- Model Summary ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,817 (11.00 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 52.4927 - val_loss: 18.1788
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 11.7737 - val_loss: 6.1447
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.9394 - val_loss: 5.0703
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 6.2100 - val_loss: 4.8080
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 5.8075 - val_loss: 4.6334
Epoch 6/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 5.5419 - val_loss: 4.5085
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 5.3401 - val_loss: 4.4206
Epoch 8/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 5.1902 - val_loss: 4.3528
Epoch 9/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 5.0789 - val_loss: 4.2985
Epoch 10/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.9870 - val_loss: 4.2524
Epoch 11/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.9242 - val_loss: 4.2220
Epoch 12/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step -